# Chained_crossing (Block C, C6) - loading the tables and the figures

This notebook **computes nothing**. It loads the csvs `aggregate.py` and
`tables.py` wrote and displays the figures `plot.py` drew, in the order the
README tells the story. Every number shown here is read from a file on disk.

Run order that produced them:

```bash
PY=/data/bfys/gscriven/conda/envs/TE/bin/python
export PYTHONNOUSERSITE=1
$PY build_tracks.py          # the 1,000 crossings, once
$PY measure_timing.py        # what a job costs
$PY make_jobs.py && condor_submit condor/jobs_chain.sub
$PY aggregate.py             # the records -> the three long tables
$PY tables.py && $PY plot.py # the display tables, the ranking, the figures
```

In [ ]:
import json
import os

import pandas as pd
from IPython.display import Image, Markdown, display

R = "results"
F = "figures"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)


def show(path, **kw):
    if os.path.exists(path):
        display(Image(filename=path, **kw))
    else:
        display(Markdown("*%s has not been drawn yet*" % path))

## The track list, and the two floors it carries

In [ ]:
meta = json.load(open(os.path.join(R, "chain_tracks_meta.json")))
print(json.dumps(meta, indent=1)[:2500])
tracks = pd.read_csv(os.path.join(R, "chain_tracks.csv"))
display(tracks.head())
print(tracks.describe()[["dz_mm", "p_GeV", "hit_vs_reference_um"]])

## What a job cost, and what the cluster did

In [ ]:
print(json.dumps(json.load(open(os.path.join(R, "timing.json"))), indent=1))
landed = pd.read_csv(os.path.join(R, "landed.csv"))
print("%d records landed" % len(landed))
display(landed.groupby(["depth", "width"])["wall_s"].describe()[
    ["count", "min", "50%", "max"]])
pending = pd.read_csv(os.path.join(R, "pending.csv"))
print("%d pending" % len(pending))

## The reference rows every table is read against

In [ ]:
refs = pd.read_csv(os.path.join(R, "reference_rows.csv"))
display(refs[(refs.direction == "all") & (refs.component == "max_xy")
             & (refs.reference == "fine")])

## The chained-crossing tables

Cells are `median [min-max over seeds]` of max(|dx|, |dy|) at the far plane
against the fine reference, in micrometres.

In [ ]:
for arch in ("4x64", "8x256"):
    p = os.path.join(R, "chained_table_%s.csv" % arch)
    if os.path.exists(p):
        display(Markdown("### %s, both directions pooled" % arch))
        display(pd.read_csv(p))

In [ ]:
for suffix in ("_x", "_y", "_tx", "_ty", "_hit"):
    p = os.path.join(R, "chained_table_4x64%s.csv" % suffix)
    if os.path.exists(p):
        display(Markdown("### 4x64%s" % suffix))
        display(pd.read_csv(p))

## The single step, per component

In [ ]:
for comp in ("x", "y", "tx", "ty", "max_xy"):
    p = os.path.join(R, "single_step_table_4x64_%s.csv" % comp)
    if os.path.exists(p):
        display(Markdown("### 4x64, component %s" % comp))
        display(pd.read_csv(p))

## The ranking

In [ ]:
rank = pd.read_csv(os.path.join(R, "ranking.csv"))
display(rank.head(15))
display(pd.read_csv(os.path.join(R, "ranking_extremes.csv")))
display(rank.groupby("best_column").size().rename("networks whose best "
                                                  "column this is"))

## How the error grows along the chain

In [ ]:
for arch in ("4x64", "8x256"):
    show(os.path.join(F, "chain_growth_%s.png" % arch), width=950)

In [ ]:
for arch in ("4x64", "8x256"):
    show(os.path.join(F, "chained_heatmap_%s.png" % arch), width=1100)

## The twelve ranked networks

In [ ]:
ext = pd.read_csv(os.path.join(R, "ranking_extremes.csv"))
for _, r in ext.iterrows():
    display(Markdown("### %s - %s %s (rank %d, %.4g um at %s)"
                     % (r["tag"], r["arm"], r["group"], r["rank"],
                        r["best_median_um"], r["best_column"])))
    show(os.path.join(F, "mini_fig3_%s.png" % r["tag"]), width=1000)
    show(os.path.join(F, "mini_fig2_%s.png" % r["tag"]), width=1200)
    show(os.path.join(F, "components_%s.png" % r["tag"]), width=800)

In [ ]:
show(os.path.join(F, "best_vs_worst_components.png"), width=1200)